# How to Create a Note Alignment

This notebook demonstrates the essential pattern for aligning a performance
with a score using {{< glossary MatchClaim >}} objects in an
{{< glossary AlignmentBundle >}}.

**What you will learn:**

1. Match performance notes to score notes by shared attributes (pitch, staff),
   giving each matched position a {{< glossary Projection >}}
2. Create an `AlignmentBundle` with performance and score groups
3. Query coordinates across both using `get_matchstamp_at()`
4. Create `MatchLine` objects from both directions
5. Export a `MatchLine` to the Vienna `.match` format
6. Opt into converted coordinates in a MatchStamp with `conversion_maps=True`

## TL;DR

```python
result = match_notes_by_attributes(
    perf_df, score_df, ["pitch", "staff"], ...,
    source_timeline=performance, target_timeline=score,
)
bundle.add_match_claims(result.match_claims)
stamp = bundle.get_matchstamp_at(78.0, "clt_1")  # quarterbeat 78 -> seconds
```

## 1. Setup

In [1]:
import tempfile

import pandas as pd

from timetoalign import Ms3Loader, TimelineGroup, TimeUnit
from timetoalign.alignment import AlignmentBundle, MatchLine
from timetoalign.alignment.match_format import MatchFileContext
from timetoalign.alignment.matching import (
    match_notes_by_attributes,
    prepare_abc_notes_for_matching,
    prepare_eep_notes_for_matching,
)
from timetoalign.core.enums import FlowMode
from timetoalign.loader.physical.eep_notes import EepNotesLoader
from timetoalign.testdata import ensure_data
from timetoalign.timelines.flow import create_unfolded_timeline

DATA_DIR = ensure_data("score") / "beethoven_op18-4iv_multimodal"
NORMAL_DIR = DATA_DIR / "StringQuartetEEP_I_Normal"
ABC_DIR = DATA_DIR / "ABC"

## 2. Load the Performance and the Score

The performance notes come from `.notes` files (EEP format with timestamps
in seconds); the loader turns them into the performance timeline. The score
is the ABC edition played through: the loader reads the folded score with
its repeats, and `create_unfolded_timeline()` lays it out in playthrough
order. That is the axis on which the pre-unfolded TSV of score notes counts
its quarterbeats.

In [2]:
# Performance notes and timeline from the Normal recording
eep_loader = EepNotesLoader()
eep_loader.load(*sorted(NORMAL_DIR.glob("*_align_*.notes")))
eep_df = eep_loader.events.to_dataframe()
perf_tl = eep_loader.create_timeline(uid="cpt_1")

# The score timeline, played through (quarterbeats)
score_loader = Ms3Loader.from_file(
    ABC_DIR / "n04op18-4_04.notes.tsv",
    ABC_DIR / "n04op18-4_04.measures.tsv",
)
flow_controller = score_loader.create_flow_controller()
clt1 = create_unfolded_timeline(
    score_loader.create_timeline(),
    flow_controller.compute_flow(FlowMode.default),
    flow_controller=flow_controller,
    uid="clt_1",
)

# Score notes from the unfolded ABC edition
abc_df = pd.read_csv(ABC_DIR / "n04op18-4_04_unfolded.notes.tsv", sep="\t")

{
    "EEP notes": len(eep_df),
    "ABC notes": len(abc_df),
    "performance length": perf_tl.length,
    "played-through score length": clt1.length,
}

{'EEP notes': 4026,
 'ABC notes': 3869,
 'performance length': Coordinate(265.899955, seconds),
 'played-through score length': Coordinate(Fraction(1116, 1), quarters)}

## 3. Prepare & Match Notes

Before matching, we drop the rests of the performance notes and explode
their chords into individual pitches. Of the score notes we keep those that
sound anew: a note without a tie, or the first note of a tie. The notes a
tie continues or ends prolong a sound already started, so a tied sound is
matched at its first note.

In [3]:
eep_prepared = prepare_eep_notes_for_matching(eep_df)
abc_prepared = prepare_abc_notes_for_matching(abc_df)

{"EEP prepared": len(eep_prepared), "ABC prepared": len(abc_prepared)}

{'EEP prepared': 3756, 'ABC prepared': 3751}

Now match by pitch name and staff number. A {{< glossary MatchClaim >}}
joins two events, but neither table names its events uniquely: the four EEP
files reuse their note IDs, one file per instrument, and a played-through
score repeats a passage's notes under the same IDs. So both sides are given
as positions (`source_coord_column`, `target_coord_column`, each on its
timeline's axis), and every matched position gets a Projection on its
timeline: an event standing there for the matched note. The notes of a
chord share one Projection, since a timeline holds one Projection per
position. The matcher returns a `MatchResult` containing the matched pairs
and the `MatchClaim` objects between the Projections.

In [4]:
match_result = match_notes_by_attributes(
    eep_prepared,
    abc_prepared,
    match_columns=["pitch", "staff"],
    source_coord_column="start",
    target_coord_column="quarterbeats_playthrough",
    source_timeline=perf_tl,  # performance timeline (seconds)
    target_timeline=clt1,  # score timeline (quarterbeats)
)

match_result.summary()

{'matched': 3741,
 'unmatched_source': 15,
 'unmatched_target': 10,
 'match_claims': 3741}

In [5]:
# The Projections the matching added, and the first claim's two events
first_claim = match_result.match_claims[0]
{
    "performance Projections": len(perf_tl.get_events(event_type="Projection")),
    "score Projections": len(clt1.get_events(event_type="Projection")),
    "first claim, performance side": perf_tl.get_event(first_claim.event_a_id),
    "first claim, score side": clt1.get_event(first_claim.event_b_id),
}

{'performance Projections': 3375,
 'score Projections': 1601,
 'first claim, performance side': {'id': 'projection_1',
  'name': None,
  'temporal_type': 'instant',
  'event_type': 'Projection',
  'start': {'value': 1.249978,
   'numerator': 2814700227510659,
   'denominator': 2251799813685248},
  'end': None,
  'duration': None,
  'pitch': None,
  'staff': None,
  'projection_of': None},
 'first claim, score side': {'id': 'projection_1',
  'name': None,
  'temporal_type': 'instant',
  'event_type': 'Projection',
  'start': {'value': 1.0, 'numerator': 1, 'denominator': 1},
  'end': None,
  'duration': None,
  'projection_of': None}}

## 4. Create AlignmentBundle

We need two {{< glossary TimelineGroup >}} objects: one for the performance,
one for the score. The `AlignmentBundle` holds both and manages cross-group
connections via {{< glossary MatchClaim >}} objects.

In [6]:
perf_group = TimelineGroup(
    id="performance",
    name="Normal Recording",
    timelines=[perf_tl],
)
perf_group

TimelineGroup(id='performance', n_timelines=1)

In [7]:
score_group = TimelineGroup(
    id="score",
    name="ABC Score",
    timelines=[clt1],
)
score_group

TimelineGroup(id='score', n_timelines=1)

In [8]:
# Create the bundle and add the match claims; the bundle gives them their ids
bundle = AlignmentBundle(name="Beethoven Op.18/4 — Simple Alignment")
bundle.add_group(perf_group)
bundle.add_group(score_group)
bundle.add_match_claims(match_result.match_claims)

bundle

AlignmentBundle(id='bundle_1', name='Beethoven Op.18/4 — Simple Alignment', timelines=2, groups=2)

In [9]:
# Display an example MatchClaim (shows event IDs, timelines, coordinates)
match_result.match_claims[0]

MatchClaim(instant: cpt_1@1.249978 seconds <-> clt_1@1 quarters)

## 5. Query Coordinates via MatchStamp

The `get_matchstamp_at()` method is the primary interface for cross-group
coordinate transfer. Given a coordinate on one timeline, it returns
the corresponding coordinates on all connected timelines.

In [10]:
# Query from the score side: quarterbeat 78 (a matched note onset)
stamp = bundle.get_matchstamp_at(78.0, "clt_1")
stamp

ID,Coordinate,Type
clt_1,78 quarters,anchor
cpt_1,18.218322 seconds,anchor
A+B,46 quarters,inferred


The played-through score is assembled from the score's sections, each a
child of `clt_1` named after its region, so the stamp also gives the
position within the section it falls in (`A+B`).

In [11]:
# The stamp shows coordinates on both timelines
{
    "score_qb": stamp.get_coordinate("clt_1"),
    "perf_seconds": stamp.get_coordinate("cpt_1"),
}

{'score_qb': IdCoordinate(Fraction(78, 1), quarters, 'clt_1'),
 'perf_seconds': IdCoordinate(18.218322, seconds, 'cpt_1')}

### Reverse lookup: performance to score

We can also query from the performance side. The claims store performance
coordinates in seconds (native EEP format).

In [12]:
# Find the score position for a performance coordinate (~100 seconds)
# Using 100.3583 which is an exact matched coordinate
stamp_rev = bundle.get_matchstamp_at(100.3583, "cpt_1")

{
    "perf_seconds": stamp_rev.get_coordinate("cpt_1"),
    "score_qb": stamp_rev.get_coordinate("clt_1"),
}

{'perf_seconds': IdCoordinate(100.3583, seconds, 'cpt_1'),
 'score_qb': IdCoordinate(Fraction(417, 1), quarters, 'clt_1')}

## 6. Create MatchLines

A {{< glossary MatchLine >}} is an ordered sequence of coordinate pairs
for a given source timeline. It is the input for WarpMap generation.

The **direction matters**: the source timeline determines the ordering.

In [13]:
# Performance-to-score: source is performance, sorted by performance time
perf_to_score = MatchLine.from_claims(
    match_result.match_claims,
    source_timeline_id="cpt_1",
)
perf_to_score

MatchLine(source='cpt_1', stamps=1473, targets=[clt_1])

In [14]:
# Score-to-performance: source is score, sorted by score position
score_to_perf = MatchLine.from_claims(
    match_result.match_claims,
    source_timeline_id="clt_1",
)
score_to_perf

MatchLine(source='clt_1', stamps=1473, targets=[cpt_1])

**When to use which direction:**

- `perf_to_score`: Use when you have a performance coordinate and want to
  find the corresponding score position. Sorted by performance time.
- `score_to_perf`: Use when you have a score coordinate and want to find
  the corresponding performance time. Sorted by score position.

Both contain the same number of stamps (one per matched note), but the
ordering and lookup direction differ.

In [15]:
# Extract typed anchors — each carries both timeline IDs and both coordinates
anchors = score_to_perf.get_alignment_anchors("cpt_1")

{
    "n_anchors": len(anchors),
    "first_anchor": anchors[0],
    "last_anchor": anchors[-1],
}

{'n_anchors': 1473,
 'first_anchor': AlignmentAnchor(clt_1@0 quarters <-> cpt_1@1 seconds),
 'last_anchor': AlignmentAnchor(clt_1@1109 quarters <-> cpt_1@264.374966 seconds)}

## 7. Export to .match Format

A {{< glossary MatchLine >}} can be exported to the Vienna `.match` file
format using `save_as()`.  The `.match` format is the standard interchange
format for note-level alignments in MIR.

To produce a rich `.match` file (with real pitch, duration, and staff
data rather than placeholders), supply a `MatchFileContext` built from
the same DataFrames used for matching.

In [16]:
ctx = MatchFileContext.from_dataframes(
    score_df=abc_prepared,
    perf_df=eep_prepared,
    match_result=match_result,
    piece="Beethoven Op.18/4-iv",
    composer="Ludwig van Beethoven",
    performer="StringQuartetEEP Normal",
)

with tempfile.TemporaryDirectory() as tmp:
    out_path = score_to_perf.save_as(f"{tmp}/alignment.match", context=ctx)
    text = out_path.read_text()

# Show the first 15 lines
for line in text.splitlines()[:15]:
    print(line)

info(matchFileVersion,1.0.0).
info(piece,Beethoven Op.18/4-iv).
info(composer,Ludwig van Beethoven).
info(performer,StringQuartetEEP Normal).
info(midiClockUnits,480).
info(midiClockRate,500000).
scoreprop(timeSignature,4/4,1:1,0,0.0000).
snote(n1,[E,b],5,1:1,0,1/2,0.0000,0.5000,[staff1])-note(n1482,75,960,1056,64,0,0).
snote(n2,[F,n],5,1:1,1/8,1/2,0.5000,1.0000,[staff1])-note(n1483,77,1056,1167,64,0,0).
snote(n3,[C,n],3,2:1,0,1,1.0000,2.0000,[staff4])-note(n1484,79,1167,1296,64,0,0).
snote(n8,[E,b],5,2:1,1/8,1/2,1.5000,2.0000,[staff1])-note(n1485,75,1296,1399,64,0,0).
snote(n9,[F,n],5,2:1,1/4,1/2,2.0000,2.5000,[staff1])-note(n1486,77,1399,1488,64,0,0).
snote(n10,[D,n],5,2:1,3/8,1/2,2.5000,3.0000,[staff1])-note(n1487,74,1488,1599,64,0,0).
snote(n11,[C,n],3,2:1,1/2,1,3.0000,4.0000,[staff4])-note(n1488,75,1599,1695,64,0,0).
snote(n16,[C,n],5,2:1,5/8,1/2,3.5000,4.0000,[staff1])-note(n1489,72,1695,1799,64,0,0).


The exported file is a valid `.match` file that can be loaded back with
`MatchfileLoader` or any tool that reads the Vienna format.

## 8. Conversion Maps in MatchStamp Display

A conversion map can be added to any timeline at any time; loaders attach
their own. Either way, `get_matchstamp_at()` leaves them out of the
{{< glossary MatchStamp >}} by default — `conversion_maps` is opt-in.

In [17]:
# Add a conversion map to the performance timeline.
from timetoalign.maps import ScalarMap

perf_tl.add_conversion_map(
    ScalarMap(
        scalar=1000, source_unit=TimeUnit.seconds, target_unit=TimeUnit.milliseconds
    )
)

In [18]:
# Default: conversion_maps=False, so none of the enabled maps show up.
bundle.get_matchstamp_at(78.0, "clt_1")

ID,Coordinate,Type
clt_1,78 quarters,anchor
cpt_1,18.218322 seconds,anchor
A+B,46 quarters,inferred


In [19]:
# conversion_maps=True adds the enabled maps, each evaluated at its own
# timeline's coordinate — ticks from the score loader's map, milliseconds
# from the map just added.
bundle.get_matchstamp_at(78.0, "clt_1", conversion_maps=True)

ID,Coordinate,Type
clt_1,78 quarters,anchor
cpt_1,18.218322 seconds,anchor
A+B,46 quarters,inferred
milliseconds,18218.322 milliseconds,cmap
ticks,22080 ticks,cmap


The same flag adds a derived column to `get_matchstamp_table()` — one
column per (timeline, enabled numeric-unit map), after the timeline
columns. A label-valued map would still show in the stamp display above
but never as a table column.

In [20]:
bundle.get_matchstamp_table([78.0], "clt_1", conversion_maps=True).column_names

['A+B', 'clt_1', 'cpt_1', 'ticks', 'milliseconds']

## Summary

> *"MatchClaims connect timelines across groups. The AlignmentBundle
> manages these connections and provides coordinate transfer via
> MatchStamps. MatchLines order these stamps for WarpMap generation."*

| Pattern | API |
|---------|-----|
| Match notes by attributes | `match_notes_by_attributes(..., source_timeline=, target_timeline=)` |
| Add claims to bundle | `bundle.add_match_claims(claims)` |
| Query by coordinate | `bundle.get_matchstamp_at(coord, tl_id)` |
| Create MatchLine | `MatchLine.from_claims(claims, source_timeline_id)` |
| Export to `.match` | `matchline.save_as("out.match", context=ctx)` |
| Reveal converted coordinates | `bundle.get_matchstamp_at(coord, tl_id, conversion_maps=True)` |